# FICA Refund Checker (prototype)
**Problem:** nonresident F-1 and J-1 students on OPT, CPT or on-campus jobs are exempt from Social Security and Medicare tax (IRC 3121(b)(19)), but payroll often withholds it anyway. Most people never get it back.

**What this does:** reads paystubs, decides paycheck by paycheck whether FICA was owed, calculates the refund, and writes the employer request letter plus the IRS fallback checklist.

**Design rule:** AI only reads documents. Tax rules are plain code with legal citations (`engine/rules.py`). All data here is **synthetic**. Not tax advice.

In [1]:
%load_ext autoreload
%autoreload 2
from datetime import date as d
from engine.rules import Profile, Paycheck
from engine.refund import compute
from engine.parser import parse_paystubs
from engine.outputs import summary, employer_letter, irs_checklist
TODAY = d(2026, 11, 18)   # fixed so results are reproducible

### Optional: turn on AI mode
Paste a Claude API key to let AI read paystubs. Press Enter with nothing to stay in **MOCK** mode (rule-based parser, no paid service). The key is hidden and never saved in this notebook.

In [2]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    key = getpass.getpass("Claude API key (Enter to skip): ")
    if key.strip():
        os.environ["ANTHROPIC_API_KEY"] = key.strip()
print("AI mode ON" if os.environ.get("ANTHROPIC_API_KEY") else "MOCK mode (no key)")

Claude API key (Enter to skip): ········
AI mode ON


## Case 1: Typical. OPT worker in year 3, FICA withheld all year
Expected: everything exempt, full refund.

In [3]:
p1 = Profile("F-1", 2026, [2024, 2025, 2026], "opt")
pcs1 = [Paycheck(d(2026, m, 1), d(2026, m, 14), d(2026, m, 17), 3000, 186, 43.5) for m in (1, 2, 3)]
r1 = compute(p1, pcs1, today=TODAY)
print(summary(r1))

Status: [R6] Year 3 of 5 as an exempt student: nonresident alien, authorized work is exempt.

Work period              Verdict  Rule      Refund     Under
2026-01-01 to 2026-01-14 exempt   R6        229.50      0.00
2026-02-01 to 2026-02-14 exempt   R6        229.50      0.00
2026-03-01 to 2026-03-14 exempt   R6        229.50      0.00

Social Security refund: $558.00
Medicare refund:        $130.50
REFUND TO CLAIM:        $688.50
Claim deadline:         2030-04-15


## Case 2: Hard. F-1 to H-1B on Oct 1, read from raw paystub text
Payroll withheld FICA during F-1 (wrong), then forgot to start it after H-1B (also wrong).
Expected: refund for F-1 work, the straddling paycheck flagged with both legal readings, under-withholding flagged separately.

In [4]:
text = open("samples/paystubs_h1b_switch.txt").read()
pcs2, errs, mode = parse_paystubs(text)
print("Parser mode:", mode, "| errors:", errs, "\n")
p2 = Profile("F-1", 2026, [2023, 2024, 2025, 2026], "opt", h1b_start=d(2026, 10, 1))
r2 = compute(p2, pcs2, today=TODAY)
print(summary(r2))

Parser mode: AI | errors: [] 

Status: [R6] Year 4 of 5 as an exempt student: nonresident alien, authorized work is exempt.

Work period              Verdict  Rule      Refund     Under
2026-05-01 to 2026-05-14 exempt   R6        229.50      0.00
2026-09-24 to 2026-10-07 split    R10         0.00      0.00
2026-10-08 to 2026-10-21 taxable  R2          0.00    229.50

Social Security refund: $186.00
Medicare refund:        $43.50
REFUND TO CLAIM:        $229.50
(Alternative reading:   $344.25, not claimed)
Claim deadline:         2030-04-15

! Ambiguous: statute vs. IRS employer guidance. Review with a tax professional.
! Payroll under-withheld $229.50 on taxable pay. Your employer is responsible for correcting this and may deduct it from future pay. Not subtracted from your refund.


## Why AI: a messy paystub from a different payroll provider
Table layout, "OASDI" instead of Social Security, "Med EE" for Medicare, current and YTD side by side. The rule-based parser fails; AI reads it. Either way, the output goes through the same validation.

In [5]:
messy = open("samples/paystub_messy.txt").read()
print("MOCK:", parse_paystubs(messy, use_ai=False))
print()
print("AI:  ", parse_paystubs(messy, use_ai=True))

MOCK: ([], ["No paychecks found. Each paystub needs a 'Pay Period' line."], 'MOCK')

AI:   ([Paycheck(period_start=datetime.date(2026, 5, 15), period_end=datetime.date(2026, 5, 28), pay_date=datetime.date(2026, 6, 5), gross=3000.0, ss_withheld=186.0, medicare_withheld=43.5)], [], 'AI')


## Case 3: Past 5 years. Year 6, in the US 200 days
Days now count toward residency. 183+ days means resident from the first day present, so FICA was owed.
Expected: no refund, plus warnings about the two exceptions the tool does not evaluate.

In [6]:
p3 = Profile("F-1", 2026, [2020, 2021, 2022, 2023, 2024], "opt",
             days_in_us=200, first_day_in_us=d(2026, 1, 1))
r3 = compute(p3, pcs1, today=TODAY)
print(summary(r3))

Status: [R9] Substantial presence met (200 weighted days): resident from 2026-01-01. Work from that date owes FICA.

Work period              Verdict  Rule      Refund     Under
2026-01-01 to 2026-01-14 taxable  R9          0.00      0.00
2026-02-01 to 2026-02-14 taxable  R9          0.00      0.00
2026-03-01 to 2026-03-14 taxable  R9          0.00      0.00

Social Security refund: $0.00
Medicare refund:        $0.00
REFUND TO CLAIM:        $0.00
Claim deadline:         2030-04-15

! If you work on campus for the school where you are enrolled at least half-time, a separate student exemption (IRC 3121(b)(10)) may still apply. Not evaluated here.
! Rare exception: a student can stay an exempt individual past 5 years by showing no intent to reside permanently (Form 8843 statement). Not evaluated here.


## Case 4: Invalid person. F-2 dependent
Expected: rejected with the legal reason. No refund.

In [7]:
r4 = compute(Profile("F-2", 2026, [2026], "opt"), pcs1, today=TODAY)
print(summary(r4))

Result: no refund. [R1] F-2 dependents are not covered by the student FICA exemption (Treas. Reg. 31.3121(b)(19)-1(a)(2)). FICA was correctly withheld.


## Case 5: Invalid input. Bad paycheck data
Expected: clear errors, nothing calculated.

In [8]:
bad = [Paycheck(d(2026, 5, 14), d(2026, 5, 1), d(2026, 5, 17), 3000, 186, 43.5),   # period ends before it starts
       Paycheck(d(2026, 6, 1), d(2026, 6, 14), d(2026, 6, 17), 100, 90, 20)]        # FICA more than wages
print(summary(compute(p1, bad, today=TODAY)))

Could not check this. Fix these first:
  - Paycheck 1: period ends before it starts.
  - Paycheck 2: FICA withheld is more than gross wages. Check the numbers.


## Case 6: Too late. Wages from 2021
Expected: refund calculated but deadline (April 15, 2025) has passed.

In [9]:
p6 = Profile("F-1", 2021, [2019, 2020, 2021], "opt")
pcs6 = [Paycheck(d(2021, 3, 1), d(2021, 3, 14), d(2021, 3, 17), 3000, 186, 43.5)]
print(summary(compute(p6, pcs6, today=TODAY)))

Status: [R6] Year 3 of 5 as an exempt student: nonresident alien, authorized work is exempt.

Work period              Verdict  Rule      Refund     Under
2021-03-01 to 2021-03-14 exempt   R6        229.50      0.00

Social Security refund: $186.00
Medicare refund:        $43.50
REFUND TO CLAIM:        $229.50
Claim deadline:         2025-04-15

! Refund deadline passed on 2025-04-15. The IRS will likely deny the claim.


## Output: what the user actually sends (from Case 2)
Step 1 per IRS: ask the employer. Step 2 only if they refuse: the IRS packet.

In [10]:
print(employer_letter(r2, "Jane Doe", "Acme Corp", "F-1 (OPT)", 2026, TODAY))

November 18, 2026

To: Payroll Department, Acme Corp
From: Jane Doe
Subject: Refund of Social Security and Medicare tax withheld in error (2026)

Hello,

Social Security and Medicare tax was withheld from my wages for work performed from May 01, 2026 to May 14, 2026. During that period I was in F-1 (OPT) status, a nonresident alien for tax purposes, doing work authorized under my status.

Under Internal Revenue Code section 3121(b)(19), these wages are not subject to Social Security or Medicare tax. IRS guidance asks employees to request repayment from the employer first.

Amounts withheld in error:
  Social Security: $186.00
  Medicare:        $43.50
  Total:           $229.50

Please refund this amount and, if it was reported for a prior year, issue a corrected Form W-2c. If you file Form 941-X to recover it, I am happy to sign the written consent you need.

If you are unable to refund it, please send me a short written statement saying so and listing any amount you have refunded or 

In [11]:
print(irs_checklist(r2, 2026))

IRS packet: use ONLY if your employer refuses or does not respond.
File one packet per tax year. Do not attach it to your tax return.

[ ] Form 843 (Claim for Refund and Request for Abatement)
      Tax period: Jan 1 - Dec 31, 2026
      Amount to be refunded: $229.50
      Type of tax: Employment (Social Security and Medicare)
      Explanation: nonresident alien student, wages exempt under IRC 3121(b)(19)
[ ] Form 8316 (refund of Social Security tax withheld on F, J or M visa wages)
[ ] Copy of Form W-2 for 2026
[ ] Copy of visa page from passport
[ ] Form I-94 (arrival and departure record)
[ ] I-20 or DS-2019, and EAD card if you were on OPT or STEM OPT
[ ] Employer's statement of refusal, OR your own signed statement explaining why you
    could not get one (include the date you asked)
[ ] Mail to: Department of the Treasury, Internal Revenue Service Center,
             Ogden, UT 84201-0038 (check IRS Publication 519 for the current address)

Deadline: postmarked by April 15, 203